In [0]:
orders_path = "s3://e-commerce-return-and-refund/raw/orders/"

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_path)
)

display(orders_df)

order_id,customer_id,product_id,seller_id,order_date,quantity,unit_price,discount,payment_method,order_status
ORD0000001,CUST18896,PROD00739,SELL0071,2026-02-25,5,19848.18,2596.06,COD,DELIVERED
ORD0000002,CUST09849,PROD00514,SELL0078,2026-03-17,1,12161.27,1527.34,DEBIT_CARD,DELIVERED
ORD0000003,CUST18040,PROD01412,SELL0078,2026-04-13,2,10329.58,3641.27,CREDIT_CARD,CANCELLED
ORD0000004,CUST12354,PROD01810,SELL0039,2026-04-10,2,21120.34,6306.56,COD,DELIVERED
ORD0000005,CUST09641,PROD01072,SELL0065,2026-06-05,1,3174.45,634.23,COD,DELIVERED
ORD0000006,CUST13253,PROD00090,SELL0089,2026-03-26,2,11930.02,2887.16,CREDIT_CARD,DELIVERED
ORD0000007,CUST07306,PROD01408,SELL0029,2026-01-20,5,13818.9,9806.93,DEBIT_CARD,DELIVERED
ORD0000008,CUST19692,PROD00046,SELL0058,2026-01-27,4,8284.63,110.38,COD,DELIVERED
ORD0000009,CUST16260,PROD01631,SELL0042,2026-05-13,3,20268.08,8579.05,NET_BANKING,DELIVERED
ORD0000010,CUST07403,PROD00156,SELL0051,2026-03-03,3,31005.78,1155.55,DEBIT_CARD,DELIVERED


In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_returns.bronze.orders")

In [0]:
datasets = {
    "customers": "s3://e-commerce-return-and-refund/raw/customers/",
    "sellers": "s3://e-commerce-return-and-refund/raw/sellers/",
    "products": "s3://e-commerce-return-and-refund/raw/products/",
    "returns": "s3://e-commerce-return-and-refund/raw/returns/",
    "refunds": "s3://e-commerce-return-and-refund/raw/refunds/"
}

for table_name, path in datasets.items():

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"ecommerce_returns.bronze.{table_name}")

    print(f"Created bronze table: {table_name}")

Created bronze table: customers
Created bronze table: sellers
Created bronze table: products
Created bronze table: returns
Created bronze table: refunds


In [0]:
for table_name in [
    "orders",
    "customers",
    "sellers",
    "products",
    "returns",
    "refunds"
]:
    count = spark.table(f"ecommerce_returns.bronze.{table_name}").count()
    print(f"{table_name}: {count:,} rows")

orders: 100,100 rows
customers: 20,000 rows
sellers: 100 rows
products: 2,000 rows
returns: 9,662 rows
refunds: 5,750 rows
